SETUP

In [0]:
import requests
import json
from datetime import datetime
from pyspark.sql.types import StructType, StructField, StringType, DateType
from config.f1_config import add_ingestion_metadata,CATALOG
headers = {"User-Agent": "MyF1Project/1.0"}

API_URL = "https://api.jolpi.ca/ergast/f1/current/results.json"
STAGING_PATH = "abfss://staging@stf1project.dfs.core.windows.net/results"
BRONZE_TABLE = "f1_workspace.bronze.results"

season = datetime.now().year
run_date = datetime.now().strftime("%Y-%m-%d")

API CALL

In [0]:
response = requests.get(API_URL, timeout=30)
response.raise_for_status()
data = response.json()

total = int(data["MRData"]["total"])
print(f"Total results available: {total}")
limit = 100
num_pages = (total // limit) + 1  

all_results = []
for page in range(num_pages):
    offset = page * limit
    paged_url = f"{API_URL}?limit={limit}&offset={offset}"

    response = requests.get(paged_url, headers=headers, timeout=30)
    response.raise_for_status()
    data = response.json()

    races = data["MRData"]["RaceTable"]["Races"]
    for race in races:
        for result in race["Results"]:
            result["round"] = race["round"]
            result["raceName"] = race["raceName"]
            all_results.append(result)

print(f"Pulled {len(all_results)} results total")

LAND DATA IN ADLS


In [0]:
raw_path = f"{STAGING_PATH}/season={season}/run_date={run_date}/results.json"

dbutils.fs.put(raw_path, json.dumps(all_results), overwrite=True)

print(f"Landed raw JSON at: {raw_path}")

FLATTEN ROWS

EXPLICIT SCHEMA


In [0]:
results_schema = StructType([
    StructField("number", StringType(), True),
    StructField("position", StringType(), True),
    StructField("positionText", StringType(), True),
    StructField("points", StringType(), True),
    StructField("grid", StringType(), True),
    StructField("laps", StringType(), True),
    StructField("status", StringType(), True),
    StructField("round", StringType(), True),
    StructField("raceName", StringType(), True),
    StructField("Driver", StructType([
        StructField("driverId", StringType(), True),
        StructField("code", StringType(), True),
        StructField("givenName", StringType(), True),
        StructField("familyName", StringType(), True),
        StructField("dateOfBirth", StringType(), True),
        StructField("nationality", StringType(), True),
    ]), True),
    StructField("Constructor", StructType([
        StructField("constructorId", StringType(), True),
        StructField("name", StringType(), True),
        StructField("nationality", StringType(), True),
    ]), True),
    StructField("FastestLap", StructType([
        StructField("rank", StringType(), True),
        StructField("lap", StringType(), True),
    ]), True),
])

READ DATA

In [0]:
df = spark.read.format("json").schema(results_schema).load(raw_path)
df = add_ingestion_metadata(df)


WRITE DATA

In [0]:
df.write.format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", True)\
    .saveAsTable(BRONZE_TABLE)
